# Fine-tune Qwen3-4B-Instruct-2507 (QLoRA) v?i train/val/test c? s?n trong data/

## 0. Cài thư viện (chạy lần đầu)

In [1]:
# !pip -q install -U "transformers>=4.53.0" "peft>=0.14.0" "trl>=0.14.0" "accelerate>=1.3.0" \
#     "bitsandbytes>=0.44.1" "datasets>=3.1.0" sentencepiece
# !pip -q install -U evaluate sacrebleu rouge-score bert-score pyvi

## 1. Chuẩn bị & kiểm tra dữ liệu

In [2]:
import json
from pathlib import Path
from collections import Counter

DATA_DIR = Path('/kaggle/input/datasets/namkhn/rag-recommend-cv/data')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH   = DATA_DIR / 'val.jsonl'
TEST_PATH  = DATA_DIR / 'test.jsonl'

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing file: {p}')

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows   = load_jsonl(VAL_PATH)
test_rows  = load_jsonl(TEST_PATH)

print('DATA_DIR :', DATA_DIR)
print('WORK_DIR :', WORK_DIR)
print('train:', len(train_rows))
print('val  :', len(val_rows))
print('test :', len(test_rows))

sample = train_rows[0]
assert isinstance(sample, dict) and 'messages' in sample, 'train.jsonl must contain {"messages": [...]} rows.'
print('Schema OK: each row has messages.')


Tổng số mẫu: 1000
Phân bố type   : Counter({'t1': 200, 't3': 200, 't2': 150, 't4': 150, 't5': 100, 't6': 100, 't7': 100})
Phân bố subtype: Counter({None: 650, 'partial_match': 165, 'partially_qualify': 112, '...': 36, 'cannot_apply': 19, 'can_apply': 18})
Phân bố split  : Counter({'train': 900, 'test': 100})


In [3]:
# Th?ng k? nhanh role trong messages

def role_counter(rows):
    c = Counter()
    for r in rows:
        for m in r.get('messages', []):
            c[m.get('role', 'unknown')] += 1
    return c

print('Train roles:', role_counter(train_rows))
print('Val roles  :', role_counter(val_rows))
print('Test roles :', role_counter(test_rows))


Sau dedup: 997 (giảm 3)


In [4]:
# D?ng tr?c ti?p train/val/test t? DATA_DIR, KH?NG split l?i.
# N?u mu?n train l?i t? ??u v? b? checkpoint c? trong WORK_DIR:
# import shutil
# shutil.rmtree(WORK_DIR / 'qwen3-4b-qlora-jd', ignore_errors=True)


{'train': 400, 'val': 39, 'test': 98}


In [5]:
# Gi? nguy?n d? li?u t?i DATA_DIR/{train,val,test}.jsonl
print('Using existing files:', TRAIN_PATH, VAL_PATH, TEST_PATH)


Đã ghi data/{train,val,test}.jsonl


## 2. Load tokenizer + model base (QLoRA 4-bit)

In [6]:
import os
from pathlib import Path

# Use writable cache in Kaggle working directory.
HF_CACHE_DIR = WORK_DIR / 'hf_cache'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR)

print('HF cache dir:', HF_CACHE_DIR.resolve())


HF cache dir: /media/urlab/KINGSTON/nlp/hf_cache


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_PATH = '/kaggle/input/datasets/namkhn/rag-recommend-cv/models--Qwen--Qwen3-4B-Instruct-2507'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, trust_remote_code=True, cache_dir=str(HF_CACHE_DIR)
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    cache_dir=str(HF_CACHE_DIR),
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print('OK - model loaded in 4-bit from local Kaggle input path')


/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:07<00:00, 50.44it/s]


OK — model loaded in 4-bit


## 3 + 4. Cấu hình LoRA & train với SFTTrainer

In [8]:
from peft import LoraConfig, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback, TrainerCallback
import inspect

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

ds_train = load_dataset('json', data_files=str(TRAIN_PATH), split='train')
ds_val   = load_dataset('json', data_files=str(VAL_PATH),   split='train')

OUTPUT_DIR = str(WORK_DIR / 'qwen3-4b-qlora-jd')

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=200,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    optim='paged_adamw_8bit',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    max_seq_length=1536,
    packing=False,
    dataset_kwargs={'skip_prepare_dataset': False},
)

sft_sig = inspect.signature(SFTConfig.__init__)
if 'eval_strategy' not in sft_sig.parameters and 'evaluation_strategy' in sft_sig.parameters:
    sft_kwargs['evaluation_strategy'] = sft_kwargs.pop('eval_strategy')
sft_kwargs = {k: v for k, v in sft_kwargs.items() if k in sft_sig.parameters}
sft_config = SFTConfig(**sft_kwargs)

class EpochQuarterProgressCallback(TrainerCallback):
    # Log progress at 25/50/75/100% of each epoch.
    def __init__(self):
        self.seen_marks = set()

    def on_step_end(self, args, state, control, **kwargs):
        if state.epoch is None:
            return
        epoch_float = float(state.epoch)
        epoch_int = int(epoch_float)
        frac = epoch_float - epoch_int
        if frac == 0 and state.global_step > 0:
            frac = 1.0
            epoch_idx = max(epoch_int, 1)
        else:
            epoch_idx = epoch_int + 1

        marks = [(0.25, 25), (0.50, 50), (0.75, 75), (0.9999, 100)]
        for th, pct in marks:
            key = (epoch_idx, pct)
            if frac >= th and key not in self.seen_marks:
                print(f'[Epoch {epoch_idx}] {pct}% complete | global_step={state.global_step}')
                self.seen_marks.add(key)

trainer_kwargs = dict(
    model=model,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    peft_config=lora_config,
    args=sft_config,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=10, early_stopping_threshold=1e-4),
        EpochQuarterProgressCallback(),
    ],
)
trainer_sig = inspect.signature(SFTTrainer.__init__)
if 'processing_class' in trainer_sig.parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)


Generating train split: 400 examples [00:00, 207664.51 examples/s]
Generating train split: 39 examples [00:00, 50847.95 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing eval dataset: 100%|██████████| 39/39 [00:00<00:00, 2031.11 examples/s]


In [9]:
trainer.train()
trainer.save_model(OUTPUT_DIR + '/best')
tokenizer.save_pretrained(OUTPUT_DIR + '/best')
print('Adapter saved →', OUTPUT_DIR + '/best')

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.804312,0.823465
75,0.756146,0.807373


/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/huggingface_hub/file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in /media/urlab/KINGSTON/nlp/hf_cache/hub/models--Qwen--Qwen3-4B-Instruct-2507. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
  warnings.warn(message)


Adapter saved → qwen3-4b-qlora-jd/best


## 5. Đánh giá trước & sau fine-tune

Chạy inference trên `test.jsonl` cho 2 model: **base** và **fine-tuned** (base + adapter), cùng decoding params (greedy, max_new_tokens=256). Sau đó tính BLEU / ROUGE-L / BERTScore.

In [10]:
# 5a. H?m inference d?ng chung
import json, gc, torch
from tqdm import tqdm

def load_test(path):
    items = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            msgs = obj['messages']
            ref = next((m['content'] for m in msgs if m['role']=='assistant'), '')
            prompt_msgs = [m for m in msgs if m['role'] != 'assistant']
            items.append({'prompt_msgs': prompt_msgs, 'reference': ref})
    return items

test_items = load_test(TEST_PATH)
print('Test size:', len(test_items))

@torch.inference_mode()
def generate_all(model, tokenizer, items, out_path, max_new_tokens=256):
    model.eval()
    preds = []
    with open(out_path, 'w', encoding='utf-8') as f:
        for it in tqdm(items):
            inputs = tokenizer.apply_chat_template(
                it['prompt_msgs'], add_generation_prompt=True,
                tokenize=True, return_dict=True, return_tensors='pt'
            ).to(model.device)
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            text = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
            preds.append(text)
            f.write(json.dumps({'pred': text, 'reference': it['reference']}, ensure_ascii=False) + '\n')
    return preds

def free():
    gc.collect(); torch.cuda.empty_cache()


Test size: 98


In [11]:
# 5b. Inference với BASE model (model hiện trong RAM đang là base + adapter sau train).
# Cách an toàn: unload, reload base sạch để chấm baseline, sau đó load lại adapter cho FT.
del trainer, model
free()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, quantization_config=bnb_config, device_map='auto',
    trust_remote_code=True, torch_dtype=torch.bfloat16,
    cache_dir=str(HF_CACHE_DIR),
)
base_model.config.use_cache = True
generate_all(base_model, tokenizer, test_items, str(WORK_DIR / 'pred_base.jsonl'))
del base_model; free()

100%|██████████| 98/98 [03:18<00:00,  2.03s/it]


In [12]:
# 5c. Inference với FINE-TUNED (base + LoRA adapter)
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, quantization_config=bnb_config, device_map='auto',
    trust_remote_code=True, torch_dtype=torch.bfloat16,
    cache_dir=str(HF_CACHE_DIR),
)
ft_model = PeftModel.from_pretrained(base_model, str(Path(OUTPUT_DIR) / 'best'))
ft_model.config.use_cache = True
generate_all(ft_model, tokenizer, test_items, str(WORK_DIR / 'pred_ft.jsonl'))
del ft_model, base_model; free()

100%|██████████| 98/98 [02:25<00:00,  1.49s/it]


In [13]:
# 5d. T?nh BLEU, ROUGE-L, BERTScore, EM, METEOR
import json
import re
import sacrebleu
import nltk
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from pyvi import ViTokenizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def load_pred(path):
    preds, refs = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            o = json.loads(line)
            preds.append(o['pred']); refs.append(o['reference'])
    return preds, refs

def normalize_text(s: str) -> str:
    s = (s or '').lower().strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def evaluate_pair(pred_path, label):
    preds, refs = load_pred(pred_path)

    bleu = sacrebleu.corpus_bleu(preds, [refs], tokenize='intl').score

    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rl = []
    for p, r in zip(preds, refs):
        p_t = ViTokenizer.tokenize(p)
        r_t = ViTokenizer.tokenize(r)
        rl.append(scorer.score(r_t, p_t)['rougeL'].fmeasure)
    rougeL = 100 * sum(rl) / len(rl)

    P, R, F1 = bert_score(preds, refs, model_type='xlm-roberta-large', lang='vi', rescale_with_baseline=False, batch_size=8)
    bs_f1 = 100 * F1.mean().item()

    em = 100 * sum(1 for p, r in zip(preds, refs) if normalize_text(p) == normalize_text(r)) / max(len(refs), 1)

    meteor_scores = []
    for p, r in zip(preds, refs):
        meteor_scores.append(nltk.translate.meteor_score.meteor_score([r.split()], p.split()))
    meteor = 100 * sum(meteor_scores) / max(len(meteor_scores), 1)

    print(f'\n== {label} ==')
    print(f'BLEU      : {bleu:.2f}')
    print(f'ROUGE-L   : {rougeL:.2f}')
    print(f'BERTScore : {bs_f1:.2f}')
    print(f'EM        : {em:.2f}')
    print(f'METEOR    : {meteor:.2f}')
    return {'BLEU': bleu, 'ROUGE-L': rougeL, 'BERTScore-F1': bs_f1, 'EM': em, 'METEOR': meteor}

res_base = evaluate_pair(str(WORK_DIR / 'pred_base.jsonl'), 'BASE (Qwen3-4B)')
res_ft   = evaluate_pair(str(WORK_DIR / 'pred_ft.jsonl'),   'FINE-TUNED (QLoRA)')

import pandas as pd
df = pd.DataFrame([res_base, res_ft], index=['Base', 'Fine-tuned'])
print('\n', df.round(2))
df.to_csv(WORK_DIR / 'eval_results.csv')


/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/huggingface_hub/file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in /media/urlab/KINGSTON/nlp/hf_cache/hub/models--xlm-roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 26439.66it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_h


== BASE (Qwen3-4B) ==
BLEU      : 15.15
ROUGE-L   : 41.26
BERTScore : 90.50


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 25664.28it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



== FINE-TUNED (QLoRA) ==
BLEU      : 25.13
ROUGE-L   : 45.91
BERTScore : 93.13

              BLEU  ROUGE-L  BERTScore-F1
Base        15.15    41.26         90.50
Fine-tuned  25.13    45.91         93.13
